# Pandas: Compare Each Value to All Subsequent Values in a Column

Learn how to efficiently compare every value in a pandas DataFrame column with all following (subsequent) values using `apply`. This is useful for tasks like finding duplicates ahead, similarity checks, or custom pairwise comparisons without full O(n²) matrices.

Based on common patterns from Stack Overflow and pandas best practices.

## 1. Import pandas

In [1]:
import pandas as pd

## 2. Create Sample DataFrame

In [4]:
val = [16, 19, 15, 19, 15]
df = pd.DataFrame({'val': val})
df

,val
0,16
1,19
2,15
3,19
4,15


## 3. Compare Each Value to All Subsequent Values

We use `df.apply()` row-wise to build a list of comparison results (1 for match, 0 otherwise) against all later rows.

In [7]:
df['match'] = df.apply(
    lambda row: [
        1 if row['val'] == df.loc[idx, 'val'] else 0
        for idx in range(row.name + 1, len(df))
    ],
    axis=1
)

df

,val,match
0,16,"[0, 0, 0, 0]"
1,19,"[0, 1, 0]"
2,15,"[0, 1]"
3,19,[0]
4,15,[]


**Expected Output:**

|    |   val | match          |
|----|-------|----------------|
|  0 |    16 | [0, 0, 0, 0]   |
|  1 |    19 | [0, 1, 0]      |
|  2 |    15 | [0, 1]         |
|  3 |    19 | [0]            |
|  4 |    15 | []             |

## 4. Custom Comparison Function (e.g., for Strings)

You can replace the equality check with any function, such as string similarity.

In [ ]:
!pip install python-Levenshtein

In [11]:
# Example with strings and a custom threshold
df_str = pd.DataFrame({'text': ['apple', 'appl', 'banana', 'apple', 'bananna']})

from Levenshtein import ratio  # pip install python-Levenshtein (or use similar)

def is_similar(a, b, threshold=0.8):
    return 1 if ratio(a, b) >= threshold else 0

df_str['similar_later'] = df_str.apply(
    lambda row: [
        is_similar(row['text'], df_str.loc[idx, 'text'])
        for idx in range(row.name + 1, len(df_str))
    ],
    axis=1
)

df_str

,text,similar_later
0,apple,"[1, 0, 1, 0]"
1,appl,"[0, 1, 0]"
2,banana,"[0, 1]"
3,apple,[0]
4,bananna,[]


## 5. Performance Notes

- This `apply` approach is clear and works well for small to medium DataFrames (thousands of rows).
- For very large DataFrames, consider vectorized alternatives with NumPy broadcasting or creating a full upper-triangular matrix:

```python
import numpy as np
arr = df['val'].values
comparisons = (arr[:, np.newaxis] == arr[np.newaxis, :])  # Full matrix
upper_tri = np.triu(comparisons, k=1)  # Exclude diagonal and lower
```

- But for per-row lists, `apply` remains the most straightforward.

In [14]:
import numpy as np
arr = df['val'].values
arr

array([16, 19, 15, 19, 15])

In [16]:
comparisons = (arr[:, np.newaxis] == arr[np.newaxis, :])  # Full matrix
upper_tri = np.triu(comparisons, k=1) 
upper_tri

array([[False, False, False, False, False],
       [False, False, False,  True, False],
       [False, False, False, False,  True],
       [False, False, False, False, False],
       [False, False, False, False, False]])

In [18]:
import pyperclip
print(df.shape)
pyperclip.copy((df_str.head(5).to_html(classes='table table-striped text-center', justify='center', index=True)))

(5, 2)
